**Mount Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Import Libraries**

In [2]:
import os
import joblib
import pandas as pd
import numpy as np

**Load Transformer Model + Scaler + Data**

In [3]:
import joblib

SCALER_PATH = "/content/drive/MyDrive/c01-price-forecasting/data/model_inputs/lstm/scaler.pkl"
scaler = joblib.load(SCALER_PATH)

print("Scaler loaded")

Scaler loaded


In [4]:
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization, Dense, Dropout, Input
from tensorflow.keras.models import Model
import tensorflow as tf

MODEL_PATH = "/content/drive/MyDrive/c01-price-forecasting/models/transformer_model.h5"

try:
    model_tr = load_model(MODEL_PATH, compile=False)
    print("Model loaded (normal)")

except:
    print("Trying fallback...")

    def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
        x = MultiHeadAttention(num_heads=num_heads, key_dim=head_size)(inputs, inputs)
        x = Dropout(dropout)(x)
        x = LayerNormalization(epsilon=1e-6)(x)

        x_ff = Dense(ff_dim, activation="relu")(x)
        x_ff = Dense(inputs.shape[-1])(x_ff)
        x = x + x_ff
        x = LayerNormalization(epsilon=1e-6)(x)

        return x

    input_shape = (12, len(scaler.feature_names_in_) - 1)

    inputs = Input(shape=input_shape)

    x = transformer_encoder(inputs, head_size=64, num_heads=4, ff_dim=128, dropout=0.1)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.2)(x)
    outputs = Dense(1)(x)

    model_tr = Model(inputs, outputs)

    model_tr.load_weights(MODEL_PATH)

    print("Model loaded (fallback rebuild)")

Trying fallback...
Model loaded (fallback rebuild)


In [5]:
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/c01-price-forecasting/data/processed/cleaned_data.csv"

df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])

print("Dataset loaded")
print(df.head())

Dataset loaded
        date district        paddy_type  min_price  max_price  avg_price  \
0 2015-03-26   Ampara  Long_Grain_White       34.0       40.0      37.14   
1 2015-04-02   Ampara  Long_Grain_White       34.0       40.0      36.40   
2 2015-04-09   Ampara  Long_Grain_White       34.0       39.0      35.68   
3 2015-04-16   Ampara  Long_Grain_White       31.0       37.0      34.97   
4 2015-04-23   Ampara  Long_Grain_White       30.0       38.0      34.03   

   production_total  price_range  week_of_year  week_sin  ...  year  month  \
0            307661          6.0            13  1.000000  ...  2015      3   
1            309335          6.0            14  0.992709  ...  2015      4   
2            309335          5.0            15  0.970942  ...  2015      4   
3            309335          6.0            16  0.935016  ...  2015      4   
4            309335          8.0            17  0.885456  ...  2015      4   

   week  lag_1  lag_2  lag_4  lag_12  rolling_mean_4  rolli

**Prepare Input**

In [6]:
def prepare_transformer_input(df_d, scaler, feature_cols, timesteps=12):

    df_last = df_d.tail(timesteps)

    X = df_last[feature_cols].values

    full = np.zeros((len(X), len(feature_cols) + 1))
    full[:, :-1] = X

    scaled = scaler.transform(full)[:, :-1]

    # Transformer expects same shape (batch, time, features)
    X_tr = scaled.reshape(1, timesteps, len(feature_cols))

    return X_tr

**Prediction Function**

In [7]:
def predict_price_transformer(district, date):

    date = pd.to_datetime(date)

    df_d = df[df['district'] == district].copy()
    df_d = df_d.sort_values('date')
    df_d = df_d[df_d['date'] < date]

    feature_cols = list(scaler.feature_names_in_[:-1])

    # Prepare input
    X_tr = prepare_transformer_input(df_d, scaler, feature_cols)

    # Predict
    y_pred_scaled = model_tr.predict(X_tr)[0][0]

    # Inverse scaling
    dummy = np.zeros((1, len(feature_cols) + 1))
    dummy[0, -1] = y_pred_scaled

    y_pred = scaler.inverse_transform(dummy)[0, -1]

    print(f"\n District: {district}")
    print(f"Date: {date}")
    print(f"Transformer Predicted Price: {y_pred:.2f}")

    return float(y_pred)

**Test Section**

In [9]:
predict_price_transformer("Anuradhapura", "2026-05-21")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step

 District: Anuradhapura
Date: 2026-05-21 00:00:00
Transformer Predicted Price: 110.14


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


110.14265833497046

**Forecast Function**

In [10]:
from datetime import timedelta

def forecast_transformer(district, start_date, weeks=4):

    start_date = pd.to_datetime(start_date)

    df_d = df[df['district'] == district].copy()
    df_d = df_d.sort_values('date')
    df_d = df_d[df_d['date'] < start_date]

    df_future = df_d.copy()
    predictions = []

    feature_cols = list(scaler.feature_names_in_[:-1])

    current_date = start_date

    for i in range(weeks):

        X_tr = prepare_transformer_input(df_future, scaler, feature_cols)

        y_pred_scaled = model_tr.predict(X_tr)[0][0]

        # Inverse scaling
        dummy = np.zeros((1, len(feature_cols) + 1))
        dummy[0, -1] = y_pred_scaled
        y_pred = scaler.inverse_transform(dummy)[0, -1]

        predictions.append({
            "date": current_date,
            "predicted_price": float(y_pred)
        })

        # Add prediction back
        new_row = df_future.iloc[-1].copy()
        new_row['date'] = current_date
        new_row['avg_price'] = y_pred

        df_future = pd.concat([df_future, pd.DataFrame([new_row])], ignore_index=True)

        current_date += timedelta(weeks=1)

    return pd.DataFrame(predictions)

**Test Section**

In [11]:
forecast_transformer("Anuradhapura", "2026-05-08", weeks=8)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step


,date,predicted_price
0,2026-05-08,110.142658
1,2026-05-15,108.646386
2,2026-05-22,106.671670
3,2026-05-29,105.688539
4,2026-06-05,104.942451
5,2026-06-12,104.178540
6,2026-06-19,103.691613
7,2026-06-26,103.278718
